In [12]:
from datetime import date

import pandas as pd
import yaml
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

CONEXION A BASE DE DATOS

In [13]:

#ABRIR ARCHIVO DE CONFIGURACION DE CONEXION A BASES DE DATOS
with open('../configuracion.yml', 'r') as f:
    configuracion = yaml.safe_load(f)
    configuracionBaseDatos= configuracion['ADVENTURE_WORKS_DB']
    configuracionBodegaDatos= configuracion['ADVENTURE_WORKS_DW']

# CREAR LAS URLs DE CONEXION

urlBaseDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBaseDatos['user'],
    password=str(configuracionBaseDatos['password']),
    host=configuracionBaseDatos['host'],
    port=configuracionBaseDatos['port'],
    database=configuracionBaseDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)

urlBodegaDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBodegaDatos['user'],
    password=str(configuracionBodegaDatos['password']),
    host=configuracionBodegaDatos['host'],
    port=configuracionBodegaDatos['port'],
    database=configuracionBodegaDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)



# CREAR EL MOTOR DE SQLALCHEMY
motorBaseDatos = create_engine(urlBaseDatos)
motorBodegaDatos = create_engine(urlBodegaDatos)

EXTRACCION

In [14]:

esquemaSalesTerritory = "Sales"

tablaSalesTerritory = pd.read_sql_table("SalesTerritory", motorBaseDatos, esquemaSalesTerritory)
# tablaSalesTerritory.head()



queryAddress="""
SELECT 
    [AddressID], 
    [AddressLine1], 
    [AddressLine2], 
    [City], 
    [StateProvinceID], 
    [PostalCode], 
    CAST(SpatialLocation AS VARCHAR(MAX)) AS SpatialLocation, 
    [rowguid],
    [ModifiedDate]
FROM Person.Address;
"""
tablaAddress = pd.read_sql_query(queryAddress, motorBaseDatos)
# tablaAddress.head()




queryStateProvince= """
SELECT 
    [StateProvinceID],
    [StateProvinceCode],
    [CountryRegionCode],
    [IsOnlyStateProvinceFlag],
    [Name],
    [TerritoryID],
    [rowguid],
    [ModifiedDate]
FROM Person.StateProvince;
"""
tablaStateProvince = pd.read_sql_query(queryStateProvince, motorBaseDatos)
# tablaStateProvince.head()








c:\Users\dange.DANGERPC\OneDrive\Escritorio\etl-aventure-works\my_env\Lib\site-packages\pandas\io\sql.py:1737: SAWarning: Did not recognize type 'Name' of column 'Name'
  self.meta.reflect(bind=self.con, only=[table_name], views=True)


TRANSFORMACION

In [15]:
addressAndStateProvince =  tablaAddress.merge(tablaStateProvince, on='StateProvinceID', how='left')
addressAndStateProvince.head()

addressAndStateProvince.drop(columns={
    'AddressID',
    'AddressLine1',
    'AddressLine2',
    'SpatialLocation',
    'rowguid_x',
    'ModifiedDate_x',
    'IsOnlyStateProvinceFlag',
    'rowguid_y',
    'ModifiedDate_y'
}, axis=1,inplace=True)

addressAndStateProvince.head()






,City,StateProvinceID,PostalCode,StateProvinceCode,CountryRegionCode,Name,TerritoryID
0,Bothell,79,98011,WA,US,Washington,1
1,Bothell,79,98011,WA,US,Washington,1
2,Bothell,79,98011,WA,US,Washington,1
3,Bothell,79,98011,WA,US,Washington,1
4,Bothell,79,98011,WA,US,Washington,1


In [16]:
dimensionGeography = addressAndStateProvince.merge(tablaSalesTerritory, on='CountryRegionCode' , how='left')
dimensionGeography.head()

,City,StateProvinceID,PostalCode,StateProvinceCode,CountryRegionCode,Name_x,TerritoryID_x,TerritoryID_y,Name_y,Group,SalesYTD,SalesLastYear,CostYTD,CostLastYear,rowguid,ModifiedDate
0,Bothell,79,98011,WA,US,Washington,1,1,Northwest,North America,7.887187e+06,3.298694e+06,0.0,0.0,43689a10-e30b-497f-b0de-11de20267ff7,2008-04-30
1,Bothell,79,98011,WA,US,Washington,1,2,Northeast,North America,2.402177e+06,3.607149e+06,0.0,0.0,00fb7309-96cc-49e2-8363-0a1ba72486f2,2008-04-30
2,Bothell,79,98011,WA,US,Washington,1,3,Central,North America,3.072175e+06,3.205014e+06,0.0,0.0,df6e7fd8-1a8d-468c-b103-ed8addb452c1,2008-04-30
3,Bothell,79,98011,WA,US,Washington,1,4,Southwest,North America,1.051085e+07,5.366576e+06,0.0,0.0,dc3e9ea0-7950-4431-9428-99dbcbc33865,2008-04-30
4,Bothell,79,98011,WA,US,Washington,1,5,Southeast,North America,2.538667e+06,3.925071e+06,0.0,0.0,6dc4165a-5e4c-42d2-809d-4344e0ac75e7,2008-04-30


In [ ]:



dimensionGeography["EnglishCountryRegionName"] = None
dimensionGeography["SpanishCountryRegionName"] = None
dimensionGeography["FrenchCountryRegionName"] = None
dimensionGeography["IpAddressLocator"] = None


dimensionGeography.drop("Name_y", axis=1 , inplace=True)
dimensionGeography.drop("Group", axis=1 , inplace=True)
dimensionGeography.drop('SalesYTD', axis=1, inplace=True)
dimensionGeography.drop('SalesLastYear', axis=1, inplace=True)
dimensionGeography.drop('CostYTD', axis=1, inplace=True)
dimensionGeography.drop('CostLastYear', axis=1, inplace=True)
dimensionGeography.drop("rowguid", axis=1 , inplace=True)
dimensionGeography.drop("ModifiedDate", axis=1 , inplace=True)
dimensionGeography.drop("TerritoryID_y", axis=1 , inplace=True)
dimensionGeography.drop("StateProvinceID", axis=1 , inplace=True)


dimensionGeography.rename(columns={
    'Name_x': 'StateProvinceName',
    'TerritoryID_x' : 'SalesTerritoryKey',
}, inplace=True)

dimensionGeography.head()

,City,PostalCode,StateProvinceCode,CountryRegionCode,StateProvinceName,SalesTerritoryKey,EnglishCountryRegionName,SpanishCountryRegionName,FrenchCountryRegionName,IpAddressLocator
0,Bothell,98011,WA,US,Washington,1,None,None,None,None
1,Bothell,98011,WA,US,Washington,1,None,None,None,None
2,Bothell,98011,WA,US,Washington,1,None,None,None,None
3,Bothell,98011,WA,US,Washington,1,None,None,None,None
4,Bothell,98011,WA,US,Washington,1,None,None,None,None


CARGAR A LA BODEGA

In [18]:
dimensionGeography.to_sql('dimensionGeography',motorBodegaDatos, if_exists='replace',index_label='GeographyKey')

136